# This is the process of categorising bundles, such that we can extract a representative subset of bundles to label.

## For instance, for our annotated samples, attaining 30 camera bundles to annotate won't help much when it comes to annotating car radio bundles. Thus, we need to also collect car radio bundles in the Electronic domain to bundle.

In [ ]:
from google.colab import userdata
my_secret_key = userdata.get('API_KEY')



if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key


In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

path = "/content/drive/MyDrive/BundleRec Data/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")

MessageError: Error: credential propagation was unsuccessful

In [1]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau
import itertools
import json
import matplotlib.pyplot as plt
from collections import Counter

In [44]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"


Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 7.00 KiB | 7.00 MiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 61 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 38.67 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Updating files: 100% (62/62), done.


In [6]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

In [8]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])


This device is a plug-and-play USB adapter that provides 5.1 channel surround sound capabilities to computers without the need for an internal sound card.
This is a pink swimsuit designed for girls aged 7 to 16, featuring a sweetheart neckline and suitable for swimming and beach activities.
A fragrant blend of star anise, cloves, Chinese cinnamon, Sichuan peppercorns, and ginger, this seasoning enhances a variety of dishes with its unique sweet and savory flavor profile.


In [17]:
def bundle_origin(bundle_ID, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    original_session_ID = session_bundle[k].iloc[bundle_ID]["session ID"]

    item_ids = session_item[k][session_item[k]["session ID"] == original_session_ID]["item ID"].values


    # descript_sess = [metadata[k][metadata[k]['item ID'] == item_id].index.tolist()[0] for item_id in item_ids]

    descript_sess = [
    metadata[k][metadata[k]['item ID'] == item_id].index[0]
    for item_id in item_ids
    if (metadata[k]['item ID'] == item_id).any()
]


    return descript_sess


def bundle_str_generator(bundle_ID, domain, desc):


    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    bundle_list = bundle_items_list[k]

    descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_items =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

    bundle_descriptions = [desc[i] for i in descript_ids]

    bundle_categories = [metadata[k].iloc[i]['categories'] for i in descript_ids]


    bundle_str = ""

    for i in range(len(bundle_items)):
        bundle_str = bundle_str + f"{i+1}. " + bundle_items[i] + ": " + bundle_descriptions[i] + "\n"

    return bundle_str, bundle_categories

In [16]:
def extract_json_simple_replace(response_text):
    """
    Extracts a JSON object from a string that has a "===JSON_START===" separator.

    This function isolates the JSON by finding the first '{' and last '}'
    to ensure it works correctly even with markdown fences or extra whitespace.

    Args:
        response_text (str): The full string containing the separator and JSON.

    Returns:
        dict: The parsed JSON object as a Python dictionary, or None if an error occurs.
    """
    try:
        # 1. Get the text after the separator
        json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries of the JSON object
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        # 3. Slice the string to get only the valid JSON
        # This will fail gracefully in the json.loads() if a brace isn't found
        json_string = json_part[first_brace : last_brace + 1]

        # 4. Parse the clean string
        parsed_json = json.loads(json_string)
        return parsed_json

    except IndexError:
        print("Error: The separator '===JSON_START===' was not found.")
        return None
    except json.JSONDecodeError:
        print("Error: Could not find or parse a valid JSON object after the separator.")
        return None


def get_json_list(dump):
    data = []
    for i in range(len(dump)):
        extracted_data = extract_json_simple_replace(dump[i] )
        data.append(extracted_data)
    return data

In [ ]:

async def making_a_lump_judgement(prompt, seed_value):

    # The system prompt that sets the stage for the LLM's task.
    system_prompt = "You are a Senior Research Analyst responsible for categorising bundles. Your goal is to produce ONE SINGLE RECOMMENDATION."



    response = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2, # Lower temperature for more deterministic, analytical tasks
        max_tokens=1500,
        seed=seed_value
    )

    reply = response.choices[0].message.content.strip()

    return reply



async def sending_intent_list_prompts(prompts, batch_size=10, delay=10):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            making_a_lump_judgement(d["prompts"], seed_value=(i + j))
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Category judgement batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results

def making_grouped_intents_prompts(intent_list):

    prompt_list = []
    for i in intent_list:
        user_prompt = f"""
**CONTEXT:**
You have received the following bundle intents from the electronics domain. Your goal is to distill it into a smaller amount of intents. Your main job is to make it such that as many intents as possible can be categorised within those 25 bundle categories.

**Intents:**
{i}

---
**YOUR TASK:**
Your goal is to distill all intents involved and group them into categories, such as camera + accessories or laptops. Try to be more general and return them in the order of frequency. Aka, this class of bundle, generally came up more often, than this other one. Thus, it should be at the top of the list.

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "final_categories": "- [First category of bundle]\\n- [Second category]\\n- [Third category]\\n- [More categories of bundles]"
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
electronic_intent_list = ["" for i in range(int(np.floor(len(electronics_intent)/100 + 1)))]


for i in range(len(electronics_intent)):

    electronic_intent_list[int(np.floor(i/100))] = electronic_intent_list[int(np.floor(i/100))] + "\n" + electronics_intent.iloc[i]['intent']


clothing_intent_list = ["" for i in range(int(np.floor(len(clothing_intent)/100 + 1)))]


for i in range(len(electronics_intent)):

    clothing_intent_list[int(np.floor(i/100))] = clothing_intent_list[int(np.floor(i/100))] + "\n" + clothing_intent.iloc[i]['intent']


food_intent_list = ["" for i in range(int(np.floor(len(food_intent)/100 + 1)))]


for i in range(len(food_intent)):

    food_intent_list[int(np.floor(i/100))] = food_intent_list[int(np.floor(i/100))] + "\n" + str(food_intent.iloc[i]['intent'])

# # char = "electronic"

# # electronic_intent_prompts = making_grouped_intents_prompts(electronic_intent_list)

# # print(electronic_intent_prompts[0])

# # electronic_intent_prompts = [{"prompts": prompt} for prompt in electronic_intent_prompts]

# # electronic_intent_categories = await sending_intent_list_prompts(electronic_intent_prompts, batch_size=128, delay=5)

# # with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
# #     pickle.dump(electronic_intent_categories, f)


# char = "clothing"

# clothing_intent_prompts = making_grouped_intents_prompts(clothing_intent_list)

# print(clothing_intent_prompts[0])

# clothing_intent_prompts = [{"prompts": prompt} for prompt in clothing_intent_prompts]

# clothing_intent_categories = await sending_intent_list_prompts(clothing_intent_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
#     pickle.dump(clothing_intent_categories, f)


# char = "food"

# food_intent_prompts = making_grouped_intents_prompts(food_intent_list)

# print(food_intent_prompts[0])

# food_intent_prompts = [{"prompts": prompt} for prompt in food_intent_prompts]

# food_intent_categories = await sending_intent_list_prompts(food_intent_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
#     pickle.dump(food_intent_categories, f)



In [18]:
char = "electronic"

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_{char}.pkl", 'rb') as f:
    electronic_intent_guys = pickle.load(f)

char = "clothing"

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_{char}.pkl", 'rb') as f:
    clothing_intent_guys = pickle.load(f)

char = "food"

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_{char}.pkl", 'rb') as f:
    food_intent_guys = pickle.load(f)

electronic_intent_json = get_json_list(electronic_intent_guys)

clothing_intent_json = get_json_list(clothing_intent_guys)

food_intent_json = get_json_list(food_intent_guys)

In [19]:
final_electronics_intent_list = ""

for i in range(len(electronic_intent_json)):
    final_electronics_intent_list = final_electronics_intent_list + "\n" + electronic_intent_json[i]["final_categories"]

final_clothing_intent_list = ""

for i in range(len(clothing_intent_json)):
    final_clothing_intent_list = final_clothing_intent_list + "\n" + clothing_intent_json[i]["final_categories"]

final_food_intent_list = ""

for i in range(len(food_intent_json)):
    final_food_intent_list = final_food_intent_list + "\n" + food_intent_json[i]["final_categories"]

# # char = "electronic"

# # final_electronic_intent_prompts = making_grouped_intents_prompts([final_electronic_intent_list])

# # print(final_electronic_intent_prompts[0])

# # final_electronic_intent_prompts = [{"prompts": prompt} for prompt in final_electronic_intent_prompts]

# # final_electronic_intent_categories = await sending_intent_list_prompts(final_electronic_intent_prompts, batch_size=128, delay=5)

# # with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
# #     pickle.dump(final_electronic_intent_categories, f)


# char = "clothing"

# final_clothing_intent_prompts = making_grouped_intents_prompts([final_clothing_intent_list])

# print(final_clothing_intent_prompts[0])

# final_clothing_intent_prompts = [{"prompts": prompt} for prompt in final_clothing_intent_prompts]

# final_clothing_intent_categories = await sending_intent_list_prompts(final_clothing_intent_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
#     pickle.dump(final_clothing_intent_categories, f)


# char = "food"

# final_food_intent_prompts = making_grouped_intents_prompts([final_food_intent_list])

# print(final_food_intent_prompts[0])

# final_food_intent_prompts = [{"prompts": prompt} for prompt in final_food_intent_prompts]

# final_food_intent_categories = await sending_intent_list_prompts(final_food_intent_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/intent_categories_{char}.pkl", 'wb') as f:
#     pickle.dump(final_food_intent_categories, f)



# Had to run again to prune duplicate bundle intents.

In [28]:
with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_electronic.pkl", 'rb') as f:
    final_electronic_intent_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_clothing.pkl", 'rb') as f:
    final_clothing_intent_guys = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/intent_categories_food.pkl", 'rb') as f:
    final_food_intent_guys = pickle.load(f)

print(final_electronic_intent_guys[0])
print()
print(final_clothing_intent_guys[0])
print()
print(final_food_intent_guys[0])

To distill the intents into broader categories, I will analyze the frequency of similar intents and group them accordingly. The goal is to create a concise list of categories that encapsulate the various intents while maintaining a focus on the most frequently mentioned items.

1. **Camera and Accessories**: This category appears most frequently, indicating a strong interest in photography-related products, including cameras, lenses, and various accessories.
  
2. **Computers and Accessories**: This category includes laptops, desktops, and their components and peripherals. It is also mentioned multiple times, reflecting a significant focus on computing devices.

3. **Audio Equipment**: This category encompasses audio devices, sound equipment, and related accessories. It appears frequently, indicating a strong interest in audio technology.

4. **Tablets and Accessories**: Tablets and their accessories are mentioned often, showing a clear demand for portable computing devices.

5. **Stor

In [29]:
final_electronic_intent_json = get_json_list(final_electronic_intent_guys)

true_electronic_intents = final_electronic_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")

final_clothing_intent_json = get_json_list(final_clothing_intent_guys)

true_clothing_intents = final_clothing_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")

final_food_intent_json = get_json_list(final_food_intent_guys)

true_food_intents = final_food_intent_json[0]['final_categories'].replace("- [", "").replace("]", "").replace("- ", "").split("\n")


In [ ]:
def finding_electronic_categories_prompts(bundle_strings, intent_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(bundle_strings)):
        user_prompt = f"""
**CONTEXT:**
You have received the following bundle from the electronics domain. Your goal is to figure out the main category which the bundle belongs to. A bundle can belong to more than one category. However, the more specific it is the better.
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise this bundle:

Intent: {intent_list[i]}
Bundle Items:
{bundle_strings[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "Camera and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Computers and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Audio Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Tablets and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Storage Solutions": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Networking Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Mobile Devices and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Travel Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Gaming": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Home Entertainment Systems": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Miscellaneous Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Power Solutions": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cables and Connectors": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Security Systems": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Car Technology and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Photography and Camera Equipment": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Adapters and Cables": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Television and Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "AV Setup": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "GPS and Navigation Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Mobile Device Protection": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Streaming and Media": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "PC Building and Assembly": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "General Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Walkie Talkies and Communication Devices": "integer, 1 if the bundle belongs in this category, 0 otherwise"
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
def finding_clothing_categories_prompts(bundle_strings, intent_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(bundle_strings)):
        user_prompt = f"""
**CONTEXT:**
You have received the following bundle from the clothing domain. Your goal is to figure out the main category which the bundle belongs to. A bundle can belong to more than one category. However, the more specific it is the better.
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise this bundle:

Intent: {intent_list[i]}
Bundle Items:
{bundle_strings[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
    "Clothing": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Footwear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Costumes and Themed Apparel": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Lingerie and Underwear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Baby and Kids Clothing": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Activewear and Sportswear": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Fashion Accessories": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Seasonal and Thematic Products": "integer, 1 if the bundle belongs in this category, 0 otherwise",
    "Electronics": "integer, 1 if the bundle belongs in this category, 0 otherwise"
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [ ]:
def finding_food_categories_prompts(bundle_strings, intent_list, final_intent_list):

    prompt_list = []

    categories_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(final_intent_list)])
    for i in range(len(bundle_strings)):
        user_prompt = f"""
**CONTEXT:**
You have received the following bundle from the food domain. Your goal is to figure out the main category which the bundle belongs to. A bundle can belong to more than one category. However, the more specific it is the better.
**Categories:**
{categories_str}

---
**YOUR TASK:**

Categorise this bundle:

Intent: {intent_list[i]}
Bundle Items:
{bundle_strings[i]}

---

After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "Snacks": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Beverages": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cooking Ingredients": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Breakfast Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Sweets and Desserts": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Health Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Canned and Packaged Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Baby Food": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Condiments and Sauces": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Fruits and Vegetables": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Specialty Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Dried and Preserved Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Grains and Pasta": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Miscellaneous": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Gift Baskets and Food Gifts": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Dietary Specific Items": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Cooking Tools and Kitchen Goods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Sweeteners": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Nuts and Seeds": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Coffee/Tea": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Vegetables and Beans": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Health-Conscious Options": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Prepared and Ready-Made Meals": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Ethnic and Specialty Foods": "integer, 1 if the bundle belongs in this category, 0 otherwise",
  "Culinary Specialties": "integer, 1 if the bundle belongs in this category, 0 otherwise"
}}
"""

        prompt_list.append(user_prompt)

    return prompt_list

In [30]:
electronic_intent_list = [electronics_intent.iloc[i]["intent"] for i in range(len(electronics_intent))]
electronic_bundle_str = [bundle_str_generator(i, "electronic", text_electronics)[0] for i in range(len(electronics_intent))]

clothing_intent_list = [clothing_intent.iloc[i]["intent"] for i in range(len(clothing_intent))]
clothing_bundle_str = [bundle_str_generator(i, "clothing", text_clothing)[0] for i in range(len(clothing_intent))]

food_intent_list = [food_intent.iloc[i]["intent"] for i in range(len(food_intent))]
food_bundle_str = [bundle_str_generator(i, "food", text_food)[0] for i in range(len(food_intent))]


In [ ]:
# char = "electronic"

# electronic_category_prompts = finding_electronic_categories_prompts(electronic_bundle_str, electronic_intent_list, true_electronic_intents)

# print(electronic_category_prompts[0])

# electronic_category_prompts = [{"prompts": prompt} for prompt in electronic_category_prompts]

# electronic_categories = await sending_intent_list_prompts(electronic_category_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/categories_{char}.pkl", 'wb') as f:
#     pickle.dump(electronic_categories, f)


# char = "clothing"

# clothing_category_prompts = finding_clothing_categories_prompts(clothing_bundle_str, clothing_intent_list, true_clothing_intents)

# print(clothing_category_prompts[0])

# clothing_category_prompts = [{"prompts": prompt} for prompt in clothing_category_prompts]

# clothing_categories = await sending_intent_list_prompts(clothing_category_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/categories_{char}.pkl", 'wb') as f:
#     pickle.dump(clothing_categories, f)



# char = "food"

# food_category_prompts = finding_food_categories_prompts(food_bundle_str, food_intent_list, true_food_intents)

# print(food_category_prompts[0])

# food_category_prompts = [{"prompts": prompt} for prompt in food_category_prompts]

# food_categories = await sending_intent_list_prompts(food_category_prompts, batch_size=128, delay=5)

# with open(f"/content/drive/MyDrive/BundleRec Data/categories_{char}.pkl", 'wb') as f:
#     pickle.dump(food_categories, f)



In [33]:
def extract_json_from_response(response_text):
    # Find the index of the first '{' and the last '}'
    start_index = response_text.find('{')
    end_index = response_text.rfind('}')

    if start_index != -1 and end_index != -1:
        # Slice the string to get only the content between the brackets
        json_string = response_text[start_index : end_index + 1]
        return json_string
    else:
        # Return None if a valid JSON block is not found
        return None

In [34]:
with open(f"/content/LLM4BEAR/BundleRec Data/categories_electronic.pkl", 'rb') as f:
    electronic_cat_guys = pickle.load(f)


with open(f"/content/LLM4BEAR/BundleRec Data/categories_clothing.pkl", 'rb') as f:
    clothing_cat_guys = pickle.load(f)


with open(f"/content/LLM4BEAR/BundleRec Data/categories_food.pkl", 'rb') as f:
    food_cat_guys = pickle.load(f)


electronic_cat_guys = [extract_json_from_response(i) for i in electronic_cat_guys]
clothing_cat_guys = [extract_json_from_response(i) for i in clothing_cat_guys]
food_cat_guys = [extract_json_from_response(i) for i in food_cat_guys]

In [35]:
print(clothing_cat_guys[136])

{
    "Clothing": 1,
    "Footwear": 1,
    "Accessories": 1,
    "Costumes and Themed Apparel": 0,
    "Lingerie and Underwear": 0,
    "Baby and Kids Clothing": 0,
    "Activewear and Sportswear": 0,
    "Fashion Accessories": 0,
    "Seasonal and Thematic Products": 0,
    "Electronics": 1
}
```

===JSON_START===
```json
{
    "Clothing": 1,
    "Footwear": 1,
    "Accessories": 1,
    "Costumes and Themed Apparel": 0,
    "Lingerie and Underwear": 0,
    "Baby and Kids Clothing": 0,
    "Activewear and Sportswear": 0,
    "Fashion Accessories": 0,
    "Seasonal and Thematic Products": 0,
    "Electronics": 1
}


In [36]:
electronic_category_count = [[] for i in range(len(true_electronic_intents))]
clothing_category_count = [[] for i in range(len(true_clothing_intents))]
food_category_count = [[] for i in range(len(true_food_intents))]



electronic_category_keys = [
    'Camera and Accessories', 'Computers and Accessories', 'Audio Equipment',
    'Tablets and Accessories', 'Storage Solutions', 'Networking Equipment',
    'Mobile Devices and Accessories', 'Travel Accessories', 'Gaming',
    'Home Entertainment Systems', 'Miscellaneous Electronics', 'Power Solutions',
    'Cables and Connectors', 'Security Systems', 'Car Technology and Accessories',
    'Photography and Camera Equipment', 'Adapters and Cables',
    'Television and Accessories', 'AV Setup', 'GPS and Navigation Accessories',
    'Mobile Device Protection', 'Streaming and Media', 'PC Building and Assembly',
    'General Electronics', 'Walkie Talkies and Communication Devices'
]

# Assuming category_count is a list of 25 empty lists
# Example initialization: category_count = [[] for _ in category_keys]

for i in electronic_cat_guys:

    dictionary = json.loads(i)
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(electronic_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        electronic_category_count[index].append(value)



clothing_category_keys = [
    'Clothing',
    'Footwear',
    'Accessories',
    'Costumes and Themed Apparel',
    'Lingerie and Underwear',
    'Baby and Kids Clothing',
    'Activewear and Sportswear',
    'Fashion Accessories',
    'Seasonal and Thematic Products',
    'Electronics',
]

# Assuming category_count is a list of 25 empty lists
# Example initialization: category_count = [[] for _ in category_keys]

for i in clothing_cat_guys:
    try:
        dictionary = json.loads(i)
    except:
        dictionary = extract_json_simple_replace(i)
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(clothing_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        clothing_category_count[index].append(value)


food_category_keys = [
    'Snacks',
    'Beverages',
    'Cooking Ingredients',
    'Breakfast Foods',
    'Sweets and Desserts',
    'Health Foods',
    'Canned and Packaged Foods',
    'Baby Food',
    'Condiments and Sauces',
    'Fruits and Vegetables',
    'Specialty Foods',
    'Dried and Preserved Foods',
    'Grains and Pasta',
    'Miscellaneous',
    'Gift Baskets and Food Gifts',
    'Dietary Specific Items',
    'Cooking Tools and Kitchen Goods',
    'Sweeteners',
    'Nuts and Seeds',
    'Coffee/Tea',
    'Vegetables and Beans',
    'Health-Conscious Options',
    'Prepared and Ready-Made Meals',
    'Ethnic and Specialty Foods',
    'Culinary Specialties',
]

# Assuming category_count is a list of 25 empty lists
# Example initialization: category_count = [[] for _ in category_keys]

for i in food_cat_guys:

    dictionary = json.loads(i)
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(food_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        food_category_count[index].append(value)


In [37]:
electronic_first_20 = [[] for i in range(25)]

electronic_num_categories = 25

for i in range(electronic_num_categories):
    # Loop through each bundle from 0 to 1749
    for j in range(1750):
        # Once we have found 20 items for the current category, stop searching
        if len(electronic_first_20[i]) >= 20:
            break # This breaks out of the inner 'j' loop and moves to the next category 'i'

        # If the bundle 'j' belongs to category 'i', add its index to our list
        if electronic_category_count[i][j] == 1:
            electronic_first_20[i].append(j)


clothing_first_25 = [[] for i in range(10)]

clothing_num_categories = 10

for i in range(clothing_num_categories):
    # Loop through each bundle from 0 to 1749
    for j in range(len(clothing_bundles_items)):
        # Once we have found 20 items for the current category, stop searching
        if len(clothing_first_25[i]) >= 25:
            break # This breaks out of the inner 'j' loop and moves to the next category 'i'

        # If the bundle 'j' belongs to category 'i', add its index to our list
        if clothing_category_count[i][j] == 1:
            clothing_first_25[i].append(j)

food_first_20 = [[] for i in range(25)]

food_num_categories = 25

for i in range(food_num_categories):
    # Loop through each bundle from 0 to 1749
    for j in range(len(food_bundles_items)):
        # Once we have found 20 items for the current category, stop searching
        if len(food_first_20[i]) >= 20:
            break # This breaks out of the inner 'j' loop and moves to the next category 'i'

        # If the bundle 'j' belongs to category 'i', add its index to our list
        if food_category_count[i][j] == 1:
            food_first_20[i].append(j)

In [38]:
total = 0
for i in electronic_first_20:
    print(i, len(i))

    total += len(i)

print(total)

[1, 2, 4, 9, 10, 17, 23, 24, 33, 49, 50, 55, 56, 68, 69, 70, 78, 79, 80, 81] 20
[1, 6, 8, 15, 16, 51, 52, 63, 65, 71, 74, 77, 79, 86, 87, 89, 95, 102, 103, 106] 20
[4, 7, 8, 22, 23, 26, 27, 28, 29, 31, 32, 35, 36, 37, 38, 44, 62, 66, 67, 68] 20
[12, 13, 25, 30, 34, 39, 43, 46, 53, 54, 58, 63, 64, 72, 75, 79, 86, 125, 135, 150] 20
[0, 5, 6, 7, 14, 15, 16, 33, 46, 48, 53, 60, 61, 70, 71, 74, 75, 77, 80, 81] 20
[2, 4, 7, 15, 16, 31, 39, 40, 41, 46, 66, 67, 71, 74, 83, 85, 101, 102, 108, 109] 20
[1, 12, 26, 28, 43, 57, 58, 64, 72, 82, 84, 93, 110, 135, 147, 159, 160, 199, 213, 218] 20
[1, 19, 46, 147, 203, 341, 343, 396, 410, 434, 465, 612, 687, 721, 724, 732, 828, 844, 845, 887] 20
[23, 65, 137, 138, 142, 183, 211, 212, 217, 241, 295, 392, 394, 395, 431, 520, 566, 569, 572, 622] 20
[2, 6, 31, 32, 36, 60, 62, 108, 110, 122, 135, 207, 208, 232, 233, 236, 240, 245, 274, 294] 20
[66, 176, 238, 309, 374, 392, 428, 520, 566, 634, 742, 823, 835, 849, 903, 988, 995, 1019, 1036, 1127] 20
[1, 11, 1

In [39]:
total = 0
for i in clothing_first_25:
    print(i, len(i))


    total += len(i)

print(total)

[0, 1, 3, 9, 12, 13, 14, 15, 16, 23, 32, 33, 39, 40, 41, 42, 46, 47, 49, 51, 52, 54, 55, 56, 57] 25
[2, 6, 13, 26, 27, 29, 30, 31, 32, 33, 38, 39, 43, 45, 49, 52, 53, 61, 62, 63, 67, 68, 71, 72, 79] 25
[1, 2, 4, 5, 7, 8, 10, 11, 15, 17, 18, 19, 20, 21, 22, 24, 25, 27, 28, 34, 35, 36, 37, 44, 48] 25
[4, 7, 24, 42, 88, 95, 122, 148, 154, 180, 192, 209, 217, 236, 237, 267, 299, 300, 301, 302, 304, 310, 318, 324, 363] 25
[2, 52, 55, 59, 60, 82, 84, 96, 107, 123, 124, 125, 126, 127, 164, 165, 166, 181, 185, 186, 187, 206, 212, 213, 239] 25
[7, 8, 12, 13, 14, 31, 32, 39, 40, 51, 56, 57, 58, 102, 125, 126, 127, 128, 129, 130, 148, 152, 153, 161, 172] 25
[3, 15, 31, 32, 39, 52, 66, 79, 80, 81, 95, 96, 100, 125, 126, 157, 164, 165, 166, 167, 178, 179, 190, 192, 193] 25
[2, 5, 10, 11, 15, 17, 18, 20, 22, 24, 28, 34, 35, 36, 44, 48, 54, 75, 80, 81, 88, 90, 97, 103, 105] 25
[16, 51, 54, 80, 90, 151, 287, 328, 418, 465, 503, 556, 865, 884, 886, 1137, 1363, 1549, 1572, 1712, 1808, 1811, 1908] 23
[13

In [40]:
total = 0

for i in food_first_20:
    print(i, len(i))

    total += len(i)

print(total)

[0, 1, 3, 5, 6, 10, 13, 22, 23, 24, 30, 36, 37, 38, 39, 40, 49, 50, 51, 53] 20
[0, 2, 4, 7, 12, 18, 19, 20, 34, 41, 42, 43, 52, 58, 60, 61, 63, 64, 65, 78] 20
[0, 1, 2, 13, 14, 15, 16, 17, 20, 25, 27, 28, 29, 32, 33, 38, 39, 44, 45, 46] 20
[13, 14, 28, 29, 51, 65, 84, 86, 110, 122, 123, 124, 129, 131, 142, 144, 147, 153, 154, 177] 20
[0, 5, 23, 26, 29, 30, 36, 38, 40, 51, 52, 53, 57, 59, 65, 68, 69, 72, 74, 75] 20
[0, 1, 3, 7, 9, 13, 22, 36, 37, 38, 39, 41, 43, 50, 61, 73, 74, 75, 77, 80] 20
[11, 16, 31, 66, 87, 110, 111, 118, 134, 135, 159, 172, 178, 179, 184, 190, 191, 203, 204, 207] 20
[8, 9, 113, 152, 241, 259, 357, 408, 436, 465, 466, 474, 495, 496, 632, 722, 725, 839, 969, 988] 20
[16, 27, 32, 33, 56, 66, 110, 119, 149, 162, 163, 225, 267, 268, 271, 272, 274, 306, 307, 319] 20
[6, 38, 108, 134, 135, 179, 246, 481, 544, 545, 614, 701, 851, 971, 1143, 1144, 1224, 1227, 1313, 1340] 20
[54, 149, 175, 193, 221, 223, 225, 231, 234, 243, 258, 305, 306, 354, 372, 373, 377, 380, 437, 475]

In [47]:
electronic_collection = []

for i in range(20):
    for j in range(len(electronic_first_20)):
        try:
            if electronic_first_20[j][i] not in electronic_collection:
                electronic_collection.append(electronic_first_20[j][i])
        except:
            continue

clothing_collection = []

for i in range(25):
    for j in range(len(clothing_first_25)):
        try:
            if clothing_first_25[j][i] not in clothing_collection:
                clothing_collection.append(clothing_first_25[j][i])
        except:
            continue

food_collection = []

for i in range(20):
    for j in range(len(food_first_20)):
        try:
            if food_first_20[j][i] not in food_collection:
                food_collection.append(food_first_20[j][i])
        except:
            continue




In [41]:
def str_bundle_item_list(bundle_ID, domain, desc):


    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2


    bundle_list = bundle_items_list[k]

    descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_items =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

    bundle_descriptions = [desc[i] for i in descript_ids]

    bundle_categories = [metadata[k].iloc[i]['categories'] for i in descript_ids]

    bundle_str = ""

    for i in range(len(bundle_items)):
        bundle_str = bundle_str + f"({descript_ids[i]}). " + bundle_items[i] + ": " + bundle_descriptions[i] + "\n"

    return bundle_str, bundle_categories


def bundle_info(bundle_IDs, domain, desc):

    bundle_items = []
    bundle_descriptions = []
    bundle_intents = []
    item_numbers = []

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    bundle_list = bundle_items_list[k]


    for bundle_ID in bundle_IDs:

        bundle_intents.append(intents[k].iloc[bundle_ID]["intent"])

        descript_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

        item_numbers.append(descript_ids)

        bundle_item =  [metadata[k].iloc[i]['titles'] for i in descript_ids]

        bundle_items.append(bundle_item)

        bundle_description = [desc[i] for i in descript_ids]

        bundle_descriptions.append(bundle_description)


    return [bundle_items, bundle_descriptions, item_numbers, bundle_intents]

In [ ]:
electronic_bundle_info = bundle_info(electronic_collection, "electronic", text_electronics)

clothing_bundle_info = bundle_info(clothing_collection, "clothing", text_clothing)

food_bundle_info = bundle_info(food_collection, "food", text_food)

# with open(f"/content/drive/MyDrive/BundleRec Data/electronic_bundle_info.pkl", 'wb') as f:
#     pickle.dump(electronic_bundle_info, f)

# with open(f"/content/drive/MyDrive/BundleRec Data/clothing_bundle_info.pkl", 'wb') as f:
#     pickle.dump(clothing_bundle_info, f)

# with open(f"/content/drive/MyDrive/BundleRec Data/food_bundle_info.pkl", 'wb') as f:
#     pickle.dump(food_bundle_info, f)



In [45]:
with open(f"/content/LLM4BEAR/BundleRec Data/electronic_bundle_info.pkl", 'rb') as f:
    electronic_info = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/clothing_bundle_info.pkl", 'rb') as f:
    clothing_info = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/food_bundle_info.pkl", 'rb') as f:
    food_info = pickle.load(f)


electronic_bundles = [
    {"bundle_intent": electronic_info[3][i],
     "items": [{"title": electronic_info[0][i][j],
                "image": "/content/LLM4BEAR/BundleRec Data/electronics/" + str(electronic_info[2][i][j]) + ".jpg",
                "description": electronic_info[1][i][j],
                } for j in range(len(electronic_info[0][i]))]

     } for i in range(len(electronic_info[0]))
]


electronic_testing_bundles = [electronic_bundles[i] for i in range(72, len(electronic_bundles))]

clothing_info = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing_bundle_info.pkl")


clothing_bundles = [
    {"bundle_intent": clothing_info[3][i],
     "items": [{"title": clothing_info[0][i][j],
                "image": "/content/LLM4BEAR/BundleRec Data/clothing/" + str(clothing_info[2][i][j]) + ".jpg",
                "description": clothing_info[1][i][j],
                } for j in range(len(clothing_info[0][i]))]

     } for i in range(len(clothing_info[0]))
]

clothing_testing_bundles = [clothing_bundles[i] for i in range(80, len(clothing_bundles))]


food_info = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food_bundle_info.pkl")


food_bundles = [
    {"bundle_intent": food_info[3][i],
     "items": [{"title": food_info[0][i][j],
                "image": "/content/LLM4BEAR/BundleRec Data/food/" + str(food_info[2][i][j]) + ".jpg",
                "description": food_info[1][i][j],
                } for j in range(len(food_info[0][i]))]

     } for i in range(len(food_info[0]))
]

# Annotated
## Electronic: 137 Bundles
## Clothing: 151 Bundles
## Food: 150 Bundles

In [52]:
for i in range(len(electronic_collection)):
    try:

        bund_str, _ = str_bundle_item_list(electronic_collection[i], "clothing", text_electronics)
        print("Bundle", electronic_collection[i], f"| {i+1}/{len(electronic_collection)} bundles.")
        print()

        print(bund_str)

        print()

        print("-----------------------------------------------------------------")
        print()
    except:

        continue

Bundle 1 | 1/262 bundles.

(658). Pink Queen Rhinestone Leopard Print Bikini Halter Top Hipster Large: A compact, portable 2TB external hard drive designed for convenient data storage and transfer.
(1140). Women's Floral Halter Swimwear Bikini Set Padded Bra Bikini Swimsuit S~L: A smartphone-compatible video security accessory that enhances home surveillance by providing real-time monitoring capabilities.
(2862). WIIPU sexy 3 row waist belly chain/Waist Chain/bikini body chain(wiipu-D245): This is a high-capacity rechargeable lithium-ion battery designed to power Baofeng UV-5R series and BF-F8 series two-way radios.


-----------------------------------------------------------------

Bundle 23 | 6/262 bundles.

(1618). Patty Women's Polo Collar V-Neck Long Sleeve Stretch Pullover Blouse: A surround sound speaker system designed to enhance audio experiences for computers with five satellite speakers and a subwoofer.
(1991). PattyBoutik Boat Neck O Ring Cut Out Shoulder 3/4 Sleeve Blouse

In [49]:
for i in range(len(clothing_collection)):
    print("Bundle", clothing_collection[i], f"| {i+1}/{len(clothing_collection)} bundles.")
    bund_str, _ = str_bundle_item_list(clothing_collection[i], "clothing", text_clothing)
    print()

    print(bund_str)

    print()

    print("-----------------------------------------------------------------")
    print()

Bundle 0 | 1/151 bundles.

(3245). PattyBoutik Cowl Neck Backless Chain Draping Halter Top: This is a stylish halter top featuring a cowl neck, backless design, and chain draping, ideal for a fashionable and sophisticated look.
(3525). PattyBoutik Unique V Neck Shawl Collar Short Sleeve Knit Jumper Blouse Tunic Top: This short sleeve knit blouse features a unique V-neck design and shawl collar, making it a stylish and comfortable option for casual wear.


-----------------------------------------------------------------

Bundle 2 | 2/151 bundles.

(533). Ollio Women's Platform Shoe Faux Suede Stiletto Leopard High Heels Multi Color Pump: These high-heeled pumps feature a faux suede leopard print design with a platform sole and stiletto heel, ideal for adding a bold touch to any outfit.
(2862). WIIPU sexy 3 row waist belly chain/Waist Chain/bikini body chain(wiipu-D245): This is a decorative waist chain featuring three rows designed to accentuate the waist, ideal for enhancing swimwear 

In [50]:
for i in range(len(food_collection)):
    print("Bundle", food_collection[i], f"| {i+1}/{len(food_collection)} bundles.")
    bund_str, _ = str_bundle_item_list(food_collection[i], "food", text_food)
    print()

    print(bund_str)

    print()

    print("-----------------------------------------------------------------")
    print()

Bundle 0 | 1/270 bundles.

(317). Nagatanien OTONA NO FURIKAKE Mini #1 | Rice Seasoning | 37.6g ( 20 Pcs ) [ Japanese Import ]: This product is a mini-sized rice seasoning that enhances the flavor of meals with a blend of savory ingredients, ideal for adding convenience and taste to rice dishes.
(2789). Nagatanien OTONA NO FURIKAKE Mini #2 | Rice Seasoning | 34.8g ( 20 Pcs ) [ Japanese Import ]: This product is a Japanese rice seasoning that adds flavor and variety to meals, typically used as a sprinkle over rice or other dishes.
(3495). Faygo, Rock &amp; Rye Soda, 2 Liter (Pack of 8): This is a pack of eight 2-liter bottles of a unique soft drink that combines a cherry-flavored soda with a hint of cream, offering a nostalgic taste experience.
(3496). Nagatanien Ochazuke Nori , Assortment&nbsp;(Pack of 50): This product is a pack of 50 individual servings of seasoned nori seaweed, commonly used to enhance rice dishes or soups with a savory flavor.
(3497). Honey Bee Brand Premium Roaste